###  Simplified TfL API ingestion for Databricks Free Edition using Amazon S3.

In [0]:
import json
import requests
from datetime import datetime, timezone

timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%d_%H%M%S_%f"
)

tfl_key = dbutils.secrets.get(scope="api_keys", key="tfl_key2")

def get_response(url):
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        return response
    except requests.RequestException as e:
        print(f"Fehler: {e}")
        return None

In [0]:

url_line = "https://api.tfl.gov.uk/Line/Route"

response = get_response(url = url_line)

data = response.json()


# 2. JSON-Objekt in einen String umwandeln
json_string_lines = json.dumps(data)

# 3. Direkt auf S3 schreiben (ohne Spark-Metadaten)
# Funktioniert mit Unity Catalog Pfaden (s3://...) oder DBFS-Mounts (/mnt/...)
s3_path_lines = f"s3://s3bucketmartin/raw_tfl/line_data/response_{timestamp}.json"


In [0]:
tfl_ids = [line["id"] for line in data if line.get("id")]

catch_modes = []

for line in data:
    if line.get("modeName") not in catch_modes:
        catch_modes.append(line.get("modeName"))

modes = ",".join(catch_modes)

In [0]:

disruption_data = []

for i in range(len(tfl_ids)):
    url_disruption = f"https://api.tfl.gov.uk/Line/{tfl_ids[i]}/Disruption?app_key={tfl_key}"   
    response = get_response(url=url_disruption)
    if response.status_code == 200:
        data = response.json()
        if data != []:
            for item in data: item["line_id"] = str(tfl_ids[i])
            disruption_data.extend(data)
        
    else:
        print(f"Error: {response.status_code})")


json_disruption_data = json.dumps(disruption_data)

s3_path_disruption = f"s3://s3bucketmartin/raw_tfl/disruption_data/response_{timestamp}.json"


In [0]:
url_status = f"https://api.tfl.gov.uk/Line/Mode/{modes}/Status"

response = get_response(url=url_status)

data = response.json()

json_string_status = json.dumps(data)

s3_path_status = f"s3://s3bucketmartin/raw_tfl/status_data/response_{timestamp}.json"


In [0]:


dbutils.fs.put(s3_path_lines, json_string_lines, overwrite=False)

dbutils.fs.put(s3_path_disruption, json_disruption_data, overwrite=False)

dbutils.fs.put(s3_path_status, json_string_status, overwrite=False)


